# JA4: valid certificate, different upload clients


Run with the **Bash** Jupyter kernel. Every code cell is a Shell Script cell.
The kernel starts in this notebook's folder. Edit [ja4-upload.env.sh](ja4-upload.env.sh)
before running the first cell, then source it only once per run.
All settings and file paths are printed; private key contents are never printed.

This notebook adds its own WAF policy to the existing `uploadp.cptdaz.org` domain.
It does not edit any other notebook, route, mTLS setting or Edge Action.
The domain must not already have a WAF association. Do not run two copies at once.
The WAF association covers the domain, but both custom rules are limited to
`PUT` requests whose path starts with `/upload/ja4/`. There are no managed rules
or terminal `Allow` rules.

Prerequisites: deployed base and `afd-uploadp`, working public DNS, Azure CLI login,
HTTPie with `--ciphers`, jq, AzCopy, OpenSSL and the existing valid/untrusted test PKI.
Your Azure identity needs WAF/AFD deployment, Log Analytics query and Storage blob
data permissions. AzCopy accesses Storage directly with your Azure CLI identity.

Run cells in order and **stop on a failed assertion**. Do not use Run All to ignore
errors. Cloud deployment creates billable resources. Waits are visible settings,
not readiness guarantees: if propagation or ingestion is slower, repeat the
relevant request and check after waiting. Cleanup is at the end and can be run
after a partial run using the same kernel variables.

JA4 identifies a TLS client configuration, not a person, device or trusted app.
We deliberately change HTTPie's TLS settings while keeping the certificate the
same. This produces a controlled comparison and also illustrates that attackers
can change fingerprints. This is not a distributed-load or rate-limit test.
JA4-grouped rate limiting (preview) is outside the scope of this notebook.

[Microsoft JA4 documentation](https://learn.microsoft.com/azure/web-application-firewall/afds/client-fingerprint-match-condition)


## What this demo covers

Discovery of the observed JA4 values, a fingerprint-specific `Log` rule, `Block`, an unblocked client configuration, missing and untrusted client certificates, unchanged uploads outside the demo path, downloads, WAF logs, recovery and cleanup. HTTPie uses the same certificate once with its default TLS configuration and once with a restricted TLS 1.2 cipher list. The notebook requires two distinct, observed JA4 values before it enables blocking. This also shows why JA4 is a client-stack signal, not application attestation and not a replacement for mTLS.

The [dedicated WAF policy](../afd/upload-ja4.bicep) is associated only with `uploadp.cptdaz.org`. Its custom rules act only on `PUT` requests under `/upload/ja4/`. No managed rules or new routes are added. The notebook refuses to replace an existing WAF association for that domain, so do not attach another WAF policy during the demo.

Cleanup removes only this run's WAF association, WAF policy, deployment record, blobs and temporary local directory. The existing mTLS and Edge Action setup stays unchanged. If execution stops early, use the cleanup section with the same kernel variables instead of sourcing the configuration again. AzCopy uses the Azure CLI identity directly against Storage, not mTLS through Front Door. Do not commit notebook outputs or local diagnostic files.

JA4-grouped rate limiting (preview) is not part of this demo. See [client fingerprint match condition](https://learn.microsoft.com/azure/web-application-firewall/afds/client-fingerprint-match-condition).


In [ ]:
set -o pipefail
source ./ja4-upload.env.sh &&
mkdir -p "$JA4_DIR" &&
env | grep -E '^(JA4_|AZCOPY_|PAGER=)' | sort


## 1. Tools, account and certificates

Use the existing test certificates. Do not regenerate the CA for this demo.
`pipefail` keeps command errors visible without terminating the Bash kernel.
No cell defines functions or uses an `if`/`else` branch.


In [ ]:
http --version &&
jq --version &&
az version &&
azcopy --version &&
openssl version


In [ ]:
az account set --subscription "$JA4_SUB" &&
az account show --query '{subscription:id,name:name}' -o json &&
test -r "$JA4_KEY" && test -r "$JA4_BAD_CERT" && test -r "$JA4_BAD_KEY" &&
openssl verify -CAfile "$JA4_REPO_ROOT/projects/mtlsupload/pki/ca.crt" "$JA4_CERT"


In [ ]:
export JA4_LAW_ID
JA4_LAW_ID=$(az monitor log-analytics workspace show -g "$JA4_RG" -n "$JA4_LAW" --query customerId -o tsv) &&
test -n "$JA4_LAW_ID" &&
printf 'JA4_LAW_ID=%s\n' "$JA4_LAW_ID"


In [ ]:
az afd endpoint show -g "$JA4_RG" --profile-name "$JA4_PROFILE" --endpoint-name "$JA4_ENDPOINT" -o json &&
az afd custom-domain show -g "$JA4_RG" --profile-name "$JA4_PROFILE" --custom-domain-name "${JA4_HOST//./-}" -o json


## 2. Preconditions before creating cloud resources

The existing domain must resolve and accept your certificate. A HEAD request
for a nonexistent blob should return 404 (or 200 if it already exists), not a
TLS failure or 403. This check creates no blob.


In [ ]:
http --ignore-stdin --timeout=30 --pretty=none --print=h \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" HEAD "$JA4_URL/preflight.json" > "$JA4_DIR/preflight.headers" &&
cat "$JA4_DIR/preflight.headers" &&
grep -Eq '^HTTP/[0-9.]+ (200|404) ' "$JA4_DIR/preflight.headers"


Inspect existing WAF associations. The assertion refuses to replace a policy
already associated with this domain, including a previous unfinished run.
Finish that run's cleanup first. A deployment error must not be bypassed.


In [ ]:
az afd security-policy list -g "$JA4_RG" --profile-name "$JA4_PROFILE" -o json > "$JA4_DIR/associations-before.json" &&
jq . "$JA4_DIR/associations-before.json" &&
jq -e --arg domain "$JA4_DOMAIN_ID" \
  '[.[].parameters.associations[]?.domains[]?.id | ascii_downcase] | index($domain | ascii_downcase) == null' \
  "$JA4_DIR/associations-before.json"


In [ ]:
az monitor diagnostic-settings list --resource \
  "/subscriptions/$JA4_SUB/resourceGroups/$JA4_RG/providers/Microsoft.Cdn/profiles/$JA4_PROFILE" -o json > "$JA4_DIR/diagnostics.json" &&
jq -e --arg workspace "$JA4_LAW_RESOURCE_ID" \
  '[.[] | select((.workspaceId // "" | ascii_downcase) == ($workspace | ascii_downcase)) | .logs[] | select(.enabled and (.categoryGroup == "allLogs" or .category == "FrontDoorWebApplicationFirewallLog"))] | length > 0' \
  "$JA4_DIR/diagnostics.json"


In [ ]:
az storage blob list --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --auth-mode login --prefix "$JA4_PREFIX/" -o json > "$JA4_DIR/blobs-before.json" &&
jq -e 'length == 0' "$JA4_DIR/blobs-before.json" &&
jq -n --arg run "$JA4_RUN_ID" '{run:$run,message:"same certificate, different TLS configurations"}' > "$JA4_FILE" &&
jq . "$JA4_FILE"


## 3. Deploy observation only

The template initially contains one `Log` rule for demo uploads. There is no
fingerprint-specific rule yet. No existing resource is deleted or redeployed
apart from this run's own WAF policy and security association.


In [ ]:
az deployment group validate -g "$JA4_RG" --template-file "$JA4_TEMPLATE" \
  --parameters profileName="$JA4_PROFILE" hostName="$JA4_HOST" \
  wafPolicyName="$JA4_WAF" securityPolicyName="$JA4_SECURITY" -o json


In [ ]:
az deployment group create -g "$JA4_RG" -n "$JA4_DEPLOYMENT" --template-file "$JA4_TEMPLATE" \
  --parameters profileName="$JA4_PROFILE" hostName="$JA4_HOST" \
  wafPolicyName="$JA4_WAF" securityPolicyName="$JA4_SECURITY" -o json


In [ ]:
sleep "$JA4_PROPAGATION_SECONDS" &&
az rest --method get --url "$JA4_ARM/Microsoft.Network/frontdoorWebApplicationFirewallPolicies/$JA4_WAF?api-version=$JA4_WAF_API" \
  --query '{state:properties.provisioningState,settings:properties.policySettings,rules:properties.customRules.rules}' -o json


## 4. Baseline with two TLS configurations

Both uploads use the same certificate and payload. The first uses HTTPie's
default TLS settings. The second forces TLS 1.2 with two supported cipher
suites. Success must be `201 Created` in both cases.


In [ ]:
http --timeout=30 --check-status --pretty=none --print=h --cert "$JA4_CERT" --cert-key "$JA4_KEY" \
  PUT "$JA4_URL/baseline-default.json" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  Content-Type:application/json < "$JA4_FILE" | tee "$JA4_DIR/baseline-default.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/baseline-default.headers"


In [ ]:
http --timeout=30 --check-status --pretty=none --print=h --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" PUT "$JA4_URL/baseline-altered.json" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 Content-Type:application/json < "$JA4_FILE" \
  | tee "$JA4_DIR/baseline-altered.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/baseline-altered.headers"


## 5. Verify the origin with Azure CLI and AzCopy

Storage operations use your signed-in Entra identity, not the client
certificate. The downloaded payload must be byte-for-byte identical.


In [ ]:
az storage blob list --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --auth-mode login --prefix "$JA4_PREFIX/" -o json > "$JA4_DIR/baseline-blobs.json" &&
jq '[.[] | {name,bytes:.properties.contentLength}]' "$JA4_DIR/baseline-blobs.json" &&
jq -e 'length == 2' "$JA4_DIR/baseline-blobs.json"


In [ ]:
azcopy copy "$JA4_STORAGE_URL/$JA4_PREFIX/baseline-default.json" "$JA4_DIR/downloaded.json" --overwrite=true &&
cmp "$JA4_FILE" "$JA4_DIR/downloaded.json" &&
jq . "$JA4_DIR/downloaded.json"


## 6. Discover the actual fingerprints

The Front Door access log records the JA4 value of every request, so both baseline uploads are visible there.
The WAF log has no JA4 value for the observation rule, use it later only for rule and action.
An empty result is not success: wait for ingestion and repeat the query.
Do not substitute a sample fingerprint from documentation.


In [ ]:
export JA4_DISCOVERY_QUERY="AzureDiagnostics
| where TimeGenerated > ago(2h)
| where Category =~ 'FrontDoorAccessLog'
| where requestUri_s contains '$JA4_RUN_ID/'
| where requestUri_s endswith '/baseline-default.json' or requestUri_s endswith '/baseline-altered.json'
| where isnotempty(clientJA4FingerPrint_s)
| extend Client = iff(requestUri_s endswith '/baseline-default.json', 'default', 'altered')
| summarize Requests=count() by Client, Fingerprint=clientJA4FingerPrint_s"
printf '%s\n' "$JA4_DISCOVERY_QUERY"


In [ ]:
sleep "$JA4_LOG_SECONDS" &&
az monitor log-analytics query --workspace "$JA4_LAW_ID" --analytics-query "$JA4_DISCOVERY_QUERY" \
  -o json > "$JA4_DIR/fingerprints.json" &&
jq . "$JA4_DIR/fingerprints.json"


In [ ]:
export JA4_DEFAULT_FINGERPRINT JA4_BLOCK_FINGERPRINT
JA4_DEFAULT_FINGERPRINT=$(jq -er '[.[] | select(.Client == "default") | .Fingerprint] | unique | select(length == 1) | .[0]' "$JA4_DIR/fingerprints.json") &&
JA4_BLOCK_FINGERPRINT=$(jq -er '[.[] | select(.Client == "altered") | .Fingerprint] | unique | select(length == 1) | .[0]' "$JA4_DIR/fingerprints.json") &&
test "${#JA4_DEFAULT_FINGERPRINT}" -eq 36 && test "${#JA4_BLOCK_FINGERPRINT}" -eq 36 &&
test "$JA4_DEFAULT_FINGERPRINT" != "$JA4_BLOCK_FINGERPRINT" &&
printf 'Default: %s\nSelected for blocking: %s\n' "$JA4_DEFAULT_FINGERPRINT" "$JA4_BLOCK_FINGERPRINT"


## 7. Log the selected fingerprint before blocking

Deploy an exact JA4 match with `Log`. The altered client must still upload.
Inspect the matching rule in the logs before moving to enforcement.


In [ ]:
export JA4_RULE
JA4_RULE=$(jq -nc --arg fingerprint "$JA4_BLOCK_FINGERPRINT" '{fingerprint:$fingerprint,action:"Log"}') &&
az deployment group create -g "$JA4_RG" -n "$JA4_DEPLOYMENT" --template-file "$JA4_TEMPLATE" \
  --parameters profileName="$JA4_PROFILE" hostName="$JA4_HOST" wafPolicyName="$JA4_WAF" \
  securityPolicyName="$JA4_SECURITY" fingerprintRule="$JA4_RULE" -o json


In [ ]:
sleep "$JA4_PROPAGATION_SECONDS" &&
http --timeout=30 --check-status --pretty=none --print=h --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" PUT "$JA4_URL/log-only.json" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 Content-Type:application/json < "$JA4_FILE" \
  | tee "$JA4_DIR/log-only.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/log-only.headers"


In [ ]:
export JA4_EVENTS_QUERY="AzureDiagnostics
| where TimeGenerated > ago(2h)
| where Category =~ 'FrontDoorWebApplicationFirewallLog'
| where requestUri_s contains '$JA4_RUN_ID/'
| project TimeGenerated, Uri=requestUri_s, Rule=ruleName_s, Action=action_s, Fingerprint=clientJA4FingerPrint_s
| order by TimeGenerated asc"
printf '%s\n' "$JA4_EVENTS_QUERY"


In [ ]:
sleep "$JA4_LOG_SECONDS" &&
az monitor log-analytics query --workspace "$JA4_LAW_ID" --analytics-query "$JA4_EVENTS_QUERY" \
  -o json > "$JA4_DIR/log-events.json" &&
jq . "$JA4_DIR/log-events.json" &&
jq -e '[.[] | select((.Uri | endswith("/log-only.json")) and (.Rule | endswith("SelectedJa4Fingerprint")) and .Action == "Log")] | length > 0' \
  "$JA4_DIR/log-events.json"


## 8. Block only the selected JA4 on demo uploads

The WAF policy remains in Prevention mode; only the selected rule changes
from `Log` to `Block`. The response must have status 403 and our dedicated
`JA4_DEMO_BLOCKED` marker. A generic 403 is not sufficient proof.


In [ ]:
JA4_RULE=$(jq -c '.action = "Block"' <<< "$JA4_RULE") &&
az deployment group create -g "$JA4_RG" -n "$JA4_DEPLOYMENT" --template-file "$JA4_TEMPLATE" \
  --parameters profileName="$JA4_PROFILE" hostName="$JA4_HOST" wafPolicyName="$JA4_WAF" \
  securityPolicyName="$JA4_SECURITY" fingerprintRule="$JA4_RULE" -o json


In [ ]:
sleep "$JA4_PROPAGATION_SECONDS" &&
http --timeout=30 --pretty=none --print=hb --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" PUT "$JA4_URL/blocked.json" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 Content-Type:application/json < "$JA4_FILE" > "$JA4_DIR/blocked.response" &&
cat "$JA4_DIR/blocked.response" &&
grep -Eq '^HTTP/[0-9.]+ 403 ' "$JA4_DIR/blocked.response" && grep -Fq '"error":"JA4_DEMO_BLOCKED"' "$JA4_DIR/blocked.response"


In [ ]:
az storage blob exists --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_PREFIX/blocked.json" --auth-mode login -o json | jq -e '.exists == false'


## 9. Verify the limits of the rule

The default TLS configuration must still upload with the same certificate.
The blocked TLS configuration must still work outside the demo path and for
GET downloads. These checks protect the existing upload behavior.


In [ ]:
http --timeout=30 --check-status --pretty=none --print=h --cert "$JA4_CERT" --cert-key "$JA4_KEY" \
  PUT "$JA4_URL/allowed-default.json" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  Content-Type:application/json < "$JA4_FILE" | tee "$JA4_DIR/allowed-default.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/allowed-default.headers"


In [ ]:
http --timeout=30 --check-status --pretty=none --print=h --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" PUT "https://$JA4_HOST/$JA4_CONTAINER/$JA4_OUTSIDE_BLOB" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 Content-Type:application/json < "$JA4_FILE" \
  | tee "$JA4_DIR/outside.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/outside.headers"


In [ ]:
http --ignore-stdin --timeout=30 --check-status --pretty=none --print=b --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" GET "$JA4_URL/baseline-altered.json" > "$JA4_DIR/http-downloaded.json" &&
cmp "$JA4_FILE" "$JA4_DIR/http-downloaded.json" &&
jq . "$JA4_DIR/http-downloaded.json"


## 10. mTLS remains mandatory

These are intentional negative tests. HTTPie can report a TLS handshake error
or receive an HTTP 403. Capture its exit code without hiding the error, then
assert a certificate-related failure or 403 and confirm that no blob exists.
DNS failures, timeouts and generic connection failures do not pass.
Use the unblocked default TLS settings so the JA4 rule is not the reason.


In [ ]:
http --timeout=30 --pretty=none --print=hb PUT "$JA4_URL/no-certificate.json" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < "$JA4_FILE" > "$JA4_DIR/no-certificate.response" 2>&1
JA4_MTLS_EXIT=$?
printf 'HTTPie exit code: %s\n' "$JA4_MTLS_EXIT"
cat "$JA4_DIR/no-certificate.response" &&
grep -Eiq '(^HTTP/[0-9.]+ 403 |certificate required|certificate_required|bad certificate|handshake failure)' "$JA4_DIR/no-certificate.response"


In [ ]:
az storage blob exists --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_PREFIX/no-certificate.json" --auth-mode login -o json | jq -e '.exists == false'


In [ ]:
http --timeout=30 --pretty=none --print=hb --cert "$JA4_BAD_CERT" --cert-key "$JA4_BAD_KEY" \
  PUT "$JA4_URL/untrusted-certificate.json" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$JA4_FILE" > "$JA4_DIR/untrusted-certificate.response" 2>&1
JA4_MTLS_EXIT=$?
printf 'HTTPie exit code: %s\n' "$JA4_MTLS_EXIT"
cat "$JA4_DIR/untrusted-certificate.response" &&
grep -Eiq '(^HTTP/[0-9.]+ 403 |unknown ca|unknown_ca|bad certificate|certificate unknown|certificate_unknown|handshake failure)' "$JA4_DIR/untrusted-certificate.response"


In [ ]:
az storage blob exists --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_PREFIX/untrusted-certificate.json" --auth-mode login -o json | jq -e '.exists == false'


## 11. Confirm WAF enforcement in the logs

The blocked request must appear under the selected JA4 rule with action
`Block`. The WAF log does not contain the JA4 value itself, the rule name is
the proof that the selected fingerprint matched. mTLS handshake failures might not
reach the WAF and are not expected to have JA4 rule log entries.


In [ ]:
sleep "$JA4_LOG_SECONDS" &&
az monitor log-analytics query --workspace "$JA4_LAW_ID" --analytics-query "$JA4_EVENTS_QUERY" \
  -o json > "$JA4_DIR/block-events.json" &&
jq . "$JA4_DIR/block-events.json" &&
jq -e '[.[] | select((.Uri | endswith("/blocked.json")) and (.Rule | endswith("SelectedJa4Fingerprint")) and .Action == "Block")] | length > 0' \
  "$JA4_DIR/block-events.json"


## 12. Recovery

Redeploy observation only by explicitly passing `null` for the targeted
rule. The same previously blocked configuration must upload successfully.


In [ ]:
az deployment group create -g "$JA4_RG" -n "$JA4_DEPLOYMENT" --template-file "$JA4_TEMPLATE" \
  --parameters profileName="$JA4_PROFILE" hostName="$JA4_HOST" wafPolicyName="$JA4_WAF" \
  securityPolicyName="$JA4_SECURITY" fingerprintRule=null -o json


In [ ]:
sleep "$JA4_PROPAGATION_SECONDS" &&
http --timeout=30 --check-status --pretty=none --print=h --ssl=tls1.2 --ciphers "$JA4_CIPHERS" \
  --cert "$JA4_CERT" --cert-key "$JA4_KEY" PUT "$JA4_URL/recovered.json" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 Content-Type:application/json < "$JA4_FILE" \
  | tee "$JA4_DIR/recovered.headers" &&
grep -Eq '^HTTP/[0-9.]+ 201 ' "$JA4_DIR/recovered.headers"


In [ ]:
az storage blob exists --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_PREFIX/recovered.json" --auth-mode login -o json | jq -e '.exists == true'


## 13. Cleanup this run

Run these cells only when you have finished inspecting the results.
Keep the original kernel and variables, even after an earlier failure.
Never source the configuration again before cleanup: it creates new run IDs.

Blob deletion is limited to this run's directory and its one outside-path blob.
Only this run's security association and WAF policy are deleted. Other domains,
routes, Edge Actions, certificates, diagnostic settings and base resources remain.
If a resource was never created, its deletion may return NotFound; inspect the
message rather than suppressing it. Wait and retry WAF deletion if AFD has not
finished detaching the policy.


In [ ]:
[[ "$JA4_RUN_ID" =~ ^ja4-[0-9]{14}-[0-9]+$ ]] &&
test "$JA4_PREFIX" = "ja4/$JA4_RUN_ID" &&
azcopy remove "$JA4_STORAGE_URL/$JA4_PREFIX" --recursive=true


In [ ]:
test "$JA4_OUTSIDE_BLOB" = "$JA4_RUN_ID-outside.json" &&
az storage blob delete --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_OUTSIDE_BLOB" --auth-mode login -o json


In [ ]:
az resource delete --ids "$JA4_SECURITY_ID" --api-version "$JA4_AFD_API"


In [ ]:
az resource delete --ids "$JA4_WAF_ID" --api-version "$JA4_WAF_API"


In [ ]:
az deployment group delete -g "$JA4_RG" -n "$JA4_DEPLOYMENT"


In [ ]:
az afd security-policy list -g "$JA4_RG" --profile-name "$JA4_PROFILE" -o json > "$JA4_DIR/associations-after.json" &&
jq -e --arg name "$JA4_SECURITY" '[.[] | select(.name == $name)] | length == 0' "$JA4_DIR/associations-after.json" &&
az storage blob list --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --auth-mode login --prefix "$JA4_PREFIX/" -o json | jq -e 'length == 0'


In [ ]:
az storage blob exists --account-name "$JA4_STORAGE" --container-name "$JA4_CONTAINER" \
  --name "$JA4_OUTSIDE_BLOB" --auth-mode login -o json | jq -e '.exists == false' &&
az resource list -g "$JA4_RG" --resource-type Microsoft.Network/frontdoorWebApplicationFirewallPolicies \
  --query "[?name=='$JA4_WAF']" -o json | jq -e 'length == 0'


The last cell deletes the temporary payloads, response captures, downloads and
AzCopy logs/plans. Save any evidence you need elsewhere first. Clear all notebook
outputs before committing; runtime values and diagnostic data must stay local.


In [ ]:
[[ "$JA4_RUN_ID" =~ ^ja4-[0-9]{14}-[0-9]+$ ]] &&
test "$JA4_DIR" = "/tmp/$JA4_RUN_ID" &&
rm -r -- "$JA4_DIR"
